# 신종 게임 은어 및 악플 필터링 텍스트 분류기

**전체 파이프라인 (순서대로 실행)**

1. 라이브러리 설치
2. 데이터 로드 & 전처리
3. kcBERT 미세조정 (Fine-Tuning)
4. 평가 & 로컬 저장
5. 예시 문장 테스트
6. Gradio 웹 UI 실행
7. (선택) Hugging Face Hub 업로드


## 1. 라이브러리 설치

In [1]:
%pip install -q torch transformers datasets accelerate evaluate scikit-learn gradio huggingface_hub sentencepiece protobuf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00


## 2. 설정

In [2]:
from __future__ import annotations

import numpy as np
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

DATASET_NAME      = "smilegate-ai/kor_unsmile"
MODEL_NAME        = "beomi/kcbert-base"   # 한국어 댓글/채팅에 적합한 BERT
OUTPUT_DIR        = "./my-custom-model"
MAX_LENGTH        = 128
NUM_LABELS        = 2
LABEL2ID          = {"정상": 0, "악플": 1}
ID2LABEL          = {0: "정상", 1: "악플"}
PROFANITY_LIST_URL = (
    "https://raw.githubusercontent.com/Tanat05/korean-profanity-resources/main/"
    "%EB%A6%AC%EA%B7%B8%EC%98%A4%EB%B8%8C%EB%A0%88%EC%A0%84%EB%93%9C_%ED%95%84%ED%84%B0%EB%A7%81%EB%A6%AC%EC%8A%A4%ED%8A%B8_2020.txt"
)

## 3. 데이터 수집 및 전처리

In [3]:
# === 학습 스킵: my-custom-model 로드 모드에서는 불필요 ===
# def to_binary_label(example: dict) -> dict:
#     """
#     kor_unsmile은 다중 라벨(혐오 카테고리 + clean)입니다.
#       - clean == 1  → 정상(0)
#       - 그 외       → 악플(1)
#     """
#     example["label"] = 0 if example["clean"] == 1 else 1
#     return example
#
#
# raw = load_dataset(DATASET_NAME)
#
# dataset = raw.map(to_binary_label)
# dataset = dataset.rename_column("문장", "text")
# dataset = dataset.select_columns(["text", "label"])
#
# print("=== 데이터셋 미리보기 ===")
# print(dataset)
# print(dataset["train"][0])
print("데이터 로드 스킵 (저장된 모델 사용)")

dataset_infos.json: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

data/valid-00000-of-00001.parquet:   0%|          | 0.00/290k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15005 [00:00<?, ? examples/s]

Generating valid split:   0%|          | 0/3737 [00:00<?, ? examples/s]

Map:   0%|          | 0/15005 [00:00<?, ? examples/s]

Map:   0%|          | 0/3737 [00:00<?, ? examples/s]

=== 데이터셋 미리보기 ===
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 15005
    })
    valid: Dataset({
        features: ['text', 'label'],
        num_rows: 3737
    })
})
{'text': '일안하는 시간은 쉬고싶어서 그런게 아닐까', 'label': 0}


## 4. 토크나이징

In [4]:
# === 학습 스킵: my-custom-model 로드 모드에서는 불필요 ===
# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
#
#
# def tokenize_function(examples):
#     return tokenizer(
#         examples["text"],
#         truncation=True,
#         padding=False,  # 배치 패딩은 DataCollator가 처리
#         max_length=MAX_LENGTH,
#     )
#
#
# tokenized = dataset.map(
#     tokenize_function,
#     batched=True,
#     remove_columns=["text"],
# )
# print(tokenized)
print("토크나이징 스킵 (저장된 모델 사용)")

config.json:   0%|          | 0.00/619 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/15005 [00:00<?, ? examples/s]

Map:   0%|          | 0/3737 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 15005
    })
    valid: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3737
    })
})


## 5. 모델 로드

In [5]:
# === beomi/kcbert-base 대신 my-custom-model에서 직접 로드 ===
# model = AutoModelForSequenceClassification.from_pretrained(
#     MODEL_NAME,
#     num_labels=NUM_LABELS,
#     id2label=ID2LABEL,
#     label2id=LABEL2ID,
# )
# print(model.config)

model = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR)
tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
print(f"모델 로드 완료: {OUTPUT_DIR}")
print(model.config)

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: beomi/kcbert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "directionality": "bidi",
  "dtype": "float32",
  "eos_token_id": null,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "\uc815\uc0c1",
    "1": "\uc545\ud50c"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "label2id": {
    "\uc545\ud50c": 1,
    "\uc815\uc0c1": 0
  },
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 300,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "pooler_fc_size": 768,
  "pooler_num_attention_heads": 12,
  "pooler_num_fc_layers": 3,
  "pooler_size_per_head": 128,
  "pooler_type": "first_token_transform",
  "tie_word_embeddings": true,
  "transformers_version": "5.0.0",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_si

## 6. 학습

In [6]:
# === 학습 스킵: my-custom-model 로드 모드에서는 불필요 ===
# def compute_metrics(eval_pred):
#     """검증 세트에 대한 Accuracy / F1 계산"""
#     logits, labels = eval_pred
#     predictions = np.argmax(logits, axis=-1)
#     return {
#         "accuracy": accuracy_score(labels, predictions),
#         "f1": f1_score(labels, predictions, average="binary"),
#     }
#
#
# training_args = TrainingArguments(
#     output_dir="./results",
#     num_train_epochs=3,
#     per_device_train_batch_size=8,
#     per_device_eval_batch_size=8,
#     learning_rate=2e-5,
#     weight_decay=0.01,
#     eval_strategy="epoch",
#     save_strategy="epoch",
#     load_best_model_at_end=True,
#     metric_for_best_model="f1",
#     logging_steps=50,
#     report_to="none",
# )
#
# data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
#
# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=tokenized["train"],
#     eval_dataset=tokenized["valid"],
#     processing_class=tokenizer,
#     data_collator=data_collator,
#     compute_metrics=compute_metrics,
# )
#
# print("=== 학습 시작 ===")
# trainer.train()
print("학습 스킵 (저장된 모델 사용)")

=== 학습 시작 ===


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

## 7. 평가

In [ ]:
# === 학습 스킵: my-custom-model 로드 모드에서는 불필요 ===
# print("=== 검증 세트 평가 ===")
# metrics = trainer.evaluate()
# print(metrics)
print("평가 스킵 (저장된 모델 사용)")

## 8. 모델 저장

In [ ]:
# === 학습 스킵: my-custom-model 로드 모드에서는 불필요 ===
# print(f"=== 모델 저장: {OUTPUT_DIR} ===")
# trainer.save_model(OUTPUT_DIR)
# tokenizer.save_pretrained(OUTPUT_DIR)
# print("로컬 저장 완료.")
print("저장 스킵 (이미 저장된 모델 사용)")

## 9. 예시 문장 테스트

학습된 모델로 게임 채팅 예시 문장을 판정하고 결과를 표로 출력합니다.

In [ ]:
import pandas as pd
from transformers import pipeline

# 모델이 메모리에 없을 경우 대비해 다시 로드
try:
    classifier
except NameError:
    classifier = pipeline(
        "text-classification",
        model=OUTPUT_DIR,
        tokenizer=OUTPUT_DIR,
        top_k=None,
    )

test_sentences = [
    # 정상 예시
    ("오늘 저녁에 랭크 한 판 어때?",          "정상"),
    ("팀원들 수고했어 다음에도 같이 하자",      "정상"),
    ("와 방금 그 플레이 진짜 멋있었어",         "정상"),
    ("미드 로밍 한 번만 와줄 수 있어?",         "정상"),
    ("ㅋㅋ 오늘 운 없었네 다음엔 이기자",       "정상"),
    # 악플 예시
    ("너 때문에 졌잖아 진짜 못하네",            "악플"),
    ("ㅋㅋㅋ 개못하네 꺼져",                   "악플"),
    ("이런 거 왜 하냐 탈주나 해",               "악플"),
    ("니 실력이면 그냥 게임 접어",              "악플"),
    ("트롤이냐 일부러 지는 거임?",              "악플"),
]

rows = []
correct = 0
for text, expected in test_sentences:
    results = sorted(classifier(text)[0], key=lambda x: x["score"], reverse=True)
    pred  = results[0]["label"]
    score = results[0]["score"]
    hit   = "O" if pred == expected else "X"
    if hit == "O":
        correct += 1
    rows.append({"문장": text, "정답": expected, "판정": pred, "Score": f"{score:.4f}", "일치": hit})

df = pd.DataFrame(rows)
print(f"정확도: {correct}/{len(test_sentences)} ({correct/len(test_sentences)*100:.1f}%)\n")
display(df)


## 10. 비속어 리스트 기반 정확도 평가 & 시각화

리그오브레전드 필터링 리스트(2020)를 사용해 모델 성능을 측정하고 시각화합니다.

In [ ]:
%matplotlib inline
import requests
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from IPython.display import display, Image
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
)
from transformers import pipeline

# ── 한글 폰트 설정 (Colab/로컬 공통) ────────────────────────────────
def _set_korean_font():
    candidates = [
        "NanumGothic", "NanumBarunGothic", "Malgun Gothic",
        "AppleGothic", "Noto Sans KR", "Noto Sans CJK KR",
    ]
    available = {f.name for f in fm.fontManager.ttflist}
    for name in candidates:
        if name in available:
            matplotlib.rc("font", family=name)
            matplotlib.rc("axes", unicode_minus=False)
            print(f"폰트 설정: {name}")
            return
    # Colab fallback: apt-get nanum
    try:
        import subprocess, shutil
        subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum"], check=True)
        fm._load_fontmanager(try_read_cache=False)
        matplotlib.rc("font", family="NanumGothic")
        matplotlib.rc("axes", unicode_minus=False)
        print("폰트 설치 및 설정: NanumGothic")
    except Exception as e:
        print(f"한글 폰트 설정 실패 (영문으로 표시됩니다): {e}")

_set_korean_font()

# ── 비속어 리스트 다운로드 ────────────────────────────────────────────
print("비속어 리스트 다운로드 중...")
resp = requests.get(PROFANITY_LIST_URL, timeout=30)
resp.encoding = "utf-8"
raw_words = [line.strip() for line in resp.text.splitlines() if line.strip()]
# 너무 짧은 토큰(1자)은 모델 입력으로 의미 없으므로 제외
profanity_words = [w for w in raw_words if len(w) >= 2]
print(f"총 비속어 단어 수: {len(raw_words)}  (2자 이상 사용: {len(profanity_words)})")

# ── 샘플링: 최대 300개 (속도 고려) ──────────────────────────────────
import random
random.seed(42)
SAMPLE_SIZE = min(300, len(profanity_words))
sampled_profanity = random.sample(profanity_words, SAMPLE_SIZE)

# 정상 문장 샘플 (악플 수와 동일하게 맞춤)
normal_sentences = [
    "오늘 날씨 진짜 좋다", "같이 게임 한 판 더 할까?", "수고했어 오늘도",
    "저녁에 뭐 먹을지 고민이야", "다음 시즌도 화이팅", "팀원들 다들 잘했어",
    "오늘 컨디션이 좀 안 좋네", "다음엔 꼭 이기자", "연습 많이 했더니 실력이 늘었어",
    "오늘 재밌었다 또 하자", "랭크 올라가면 같이 파티하자", "미드 로밍 타이밍 좋았어",
    "오늘 운이 없었던 것 같아", "다음엔 더 잘할 수 있을 거야", "전략 바꿔보는 건 어때?",
]
normal_sampled = (normal_sentences * (SAMPLE_SIZE // len(normal_sentences) + 1))[:SAMPLE_SIZE]

# ── 전체 테스트셋 구성 ────────────────────────────────────────────────
texts  = sampled_profanity + normal_sampled
labels = ["악플"] * SAMPLE_SIZE + ["정상"] * SAMPLE_SIZE

# ── classifier 로드 (이미 있으면 재사용) ─────────────────────────────
try:
    classifier
except NameError:
    print(f"모델 로딩: {OUTPUT_DIR}")
    classifier = pipeline(
        "text-classification",
        model=OUTPUT_DIR,
        tokenizer=OUTPUT_DIR,
        top_k=None,
    )

# ── 추론 ─────────────────────────────────────────────────────────────
print(f"추론 중... (총 {len(texts)}개)")
preds = []
BATCH = 32
for i in range(0, len(texts), BATCH):
    batch = texts[i:i+BATCH]
    results = classifier(batch)
    for res in results:
        top = max(res, key=lambda x: x["score"])
        preds.append(top["label"])

# ── 지표 계산 ─────────────────────────────────────────────────────────
acc = accuracy_score(labels, preds)
f1  = f1_score(labels, preds, pos_label="악플", average="binary")
report = classification_report(labels, preds, target_names=["정상", "악플"], output_dict=True)
cm = confusion_matrix(labels, preds, labels=["악플", "정상"])

print(f"\n정확도(Accuracy): {acc:.4f}")
print(f"F1 Score (악플): {f1:.4f}")
print("\n분류 리포트:")
print(classification_report(labels, preds, target_names=["정상", "악플"]))

# ── 시각화 ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("모델 성능 평가 (비속어 리스트 기반)", fontsize=15, fontweight="bold", y=1.02)

# 1) Confusion Matrix Heatmap
ax1 = axes[0]
sns.heatmap(
    cm,
    annot=True, fmt="d", cmap="Blues",
    xticklabels=["악플(예측)", "정상(예측)"],
    yticklabels=["악플(실제)", "정상(실제)"],
    ax=ax1, linewidths=0.5, cbar_kws={"shrink": 0.8},
)
ax1.set_title("Confusion Matrix", fontsize=13, pad=12)
ax1.set_ylabel("실제 레이블", fontsize=11)
ax1.set_xlabel("예측 레이블", fontsize=11)

# 2) Per-class Precision / Recall / F1 Bar Chart
ax2 = axes[1]
classes   = ["정상", "악플"]
metrics_names = ["precision", "recall", "f1-score"]
x = np.arange(len(classes))
width = 0.25
colors = ["#4C72B0", "#DD8452", "#55A868"]
for i, (m, c) in enumerate(zip(metrics_names, colors)):
    vals = [report[cls][m] for cls in classes]
    bars = ax2.bar(x + i * width, vals, width, label=m.replace("-score", ""), color=c, alpha=0.85)
    for bar, v in zip(bars, vals):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f"{v:.2f}", ha="center", va="bottom", fontsize=9)
ax2.set_xticks(x + width)
ax2.set_xticklabels(classes, fontsize=11)
ax2.set_ylim(0, 1.15)
ax2.set_title("클래스별 Precision / Recall / F1", fontsize=13, pad=12)
ax2.set_ylabel("Score", fontsize=11)
ax2.legend(fontsize=10)
ax2.grid(axis="y", linestyle="--", alpha=0.5)

# 3) Overall Accuracy & Macro F1 Gauge Bar
ax3 = axes[2]
overall_metrics = {
    "Accuracy": acc,
    "Macro F1": report["macro avg"]["f1-score"],
    "Weighted F1": report["weighted avg"]["f1-score"],
    "F1 (악플)": f1,
}
bars = ax3.barh(
    list(overall_metrics.keys()),
    list(overall_metrics.values()),
    color=["#4C72B0", "#55A868", "#8172B2", "#DD8452"],
    alpha=0.85, height=0.5,
)
for bar, v in zip(bars, overall_metrics.values()):
    ax3.text(v + 0.01, bar.get_y() + bar.get_height()/2,
             f"{v:.4f}", va="center", fontsize=10, fontweight="bold")
ax3.set_xlim(0, 1.15)
ax3.set_title("종합 지표", fontsize=13, pad=12)
ax3.set_xlabel("Score", fontsize=11)
ax3.axvline(x=1.0, color="gray", linestyle="--", alpha=0.4)
ax3.grid(axis="x", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig("model_evaluation.png", dpi=150, bbox_inches="tight")
plt.show()
display(Image("model_evaluation.png"))
print("시각화 저장 완료: model_evaluation.png")

## 11. Gradio 웹 UI

학습 완료 후 바로 아래 셀을 실행하면 `http://127.0.0.1:7860` 에서 데모를 확인할 수 있습니다.

In [ ]:
import gradio as gr
from transformers import pipeline

print(f"모델 로딩 중: {OUTPUT_DIR}")
classifier = pipeline(
    "text-classification",
    model=OUTPUT_DIR,
    tokenizer=OUTPUT_DIR,
    top_k=None,
)
print("모델 로딩 완료.")


def predict(text: str) -> str:
    """입력 채팅 문장에 대해 정상/악플 판정과 확률을 반환합니다."""
    if not text or not text.strip():
        return "텍스트를 입력해 주세요."

    results = sorted(classifier(text)[0], key=lambda x: x["score"], reverse=True)
    top = results[0]

    lines = [
        f"판정: {top['label']}",
        f"Score: {top['score']:.4f}",
        "",
        "전체 점수:",
    ]
    for item in results:
        lines.append(f"  - {item['label']}: {item['score']:.4f}")
    return "\n".join(lines)


demo = gr.Interface(
    fn=predict,
    inputs=gr.Textbox(
        lines=3,
        label="게임 채팅 입력",
        placeholder="예: 오늘 한 판 더 할까? / 너 진짜 쓰레기네",
    ),
    outputs=gr.Textbox(label="분류 결과", lines=8),
    title="게임 채팅 악플 필터링 분류기",
    description=(
        "한국어 게임 채팅/댓글 문장을 입력하면 "
        "정상 또는 악플 여부와 확률(Score)을 출력합니다. "
        "(데이터: smilegate-ai/kor_unsmile, 모델: beomi/kcbert-base fine-tuned)"
    ),
    examples=[
        ["오늘 저녁에 랭크 한 판 어때?"],
        ["너 때문에 졌잖아 진짜 못하네"],
        ["ㅋㅋㅋ 개못하네 꺼져"],
        ["팀원들 수고했어 다음에도 같이 하자"],
    ],
)

# Colab 환경에서는 share=True 필수 - 자동으로 공개 링크(https://xxxx.gradio.live)가 생성됩니다
demo.launch(share=True)

## 11. (선택) Hugging Face Hub 업로드

1. 터미널에서 `huggingface-cli login` 실행 후 **Write** 권한 토큰 입력
2. 아래 셀 주석 해제 후 `YOUR_HF_USERNAME`을 본인 계정명으로 변경

In [ ]:
# from huggingface_hub import HfApi
#
# repo_id = "YOUR_HF_USERNAME/game-chat-abuse-filter"
# api = HfApi()
# api.upload_folder(
#     folder_path=OUTPUT_DIR,
#     repo_id=repo_id,
#     repo_type="model",
# )
# print(f"Hub 업로드 완료: https://huggingface.co/{repo_id}")